In [5]:
from datasets import load_dataset, Dataset
import pandas as pd
import numpy as np
import random as rd
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, pipeline
import torch
from tqdm import tqdm
import evaluate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from peft import LoraConfig, get_peft_model, TaskType
from evaluate import load
from sklearn.model_selection import train_test_split

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
#from trasformers 

In [7]:
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [8]:
ds_train = load_dataset("FreedomIntelligence/RAG-Instruct", split="train[:80%]")
ds_test = load_dataset("FreedomIntelligence/RAG-Instruct", split="train[80%:]")
rankings_train = np.load('/kaggle/input/xenc-scores-distilroberta/xenc_scores_train-stsb-distilroberta-base.npy')
rankings_test = np.load('/kaggle/input/xenc-scores-distilroberta/xenc_scores_test-stsb-distilroberta-base.npy')

k = 3

README.md:   0%|          | 0.00/2.64k [00:00<?, ?B/s]

rag_instruct.json:   0%|          | 0.00/296M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/40541 [00:00<?, ? examples/s]

In [ ]:
def build_df(ds, doc_rankings):
    docs = [d for d in ds['documents']]
    questions = [q for q in ds['question']]
    answers = [a for a in ds['answer']]
    data = []

    for i, (q, a) in enumerate(zip(questions, answers)):
        ranked_indices = [int(t[1]) for t in doc_rankings[i][:k]]
        top_docs = [docs[i][idx] for idx in ranked_indices]
        data.append({
            'question': q,
            'answer': a,
            'topk_documents': top_docs
        })
        if(1 == 0):
            print(i)
            print("Question: " + q + "\n")
            print("Documents:\n")
            for d in top_docs:
                print(d + "\n")
            print("Answer: " + a + "\n")
    df = pd.DataFrame(data)
    topk_df = Dataset.from_pandas(df)
    return topk_df

In [ ]:
topk_ds_train = build_df(ds_train, rankings_train)
topk_ds_testval = build_df(ds_test, rankings_test)

topk_ds_split = topk_ds_testval.train_test_split(
    test_size=0.25,
    shuffle=True,
    seed=42
)

topk_ds_val = topk_ds_split["train"]
topk_ds_test = topk_ds_split["test"] 

### Performance Evaluation

In [38]:
from torch.utils.data import DataLoader
from transformers import TextDataset, DataCollatorWithPadding
import math

def compute_perplexity(model, tokenizer, inputs, labels, device):
    model.eval()
    with torch.no_grad():
        outputs = model(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device),
            labels=labels["input_ids"].to(device)
        )
        loss = outputs.loss
    return math.exp(loss.item())

def evaluate_model(model, tokenizer, test_dataset, max_length=512):
    exact_match_metric = load("exact_match")
    squad = load("f1")
    rouge_metric = load("rouge")
    bertscore_metric = load("bertscore")
    #perplexity_metric = load("perplexity")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    batch_size = 8
    batches = [
        test_dataset.select(range(i, min(i + batch_size, len(test_dataset))))
        for i in range(0, len(test_dataset), batch_size)
    ]

    all_predictions = []
    all_references = []
    all_inputs = []

    for batch in tqdm(batches, desc="Evaluating"):
        batch_inputs = []
        references = []

        for i in range(len(batch["question"])):
            question = batch["question"][i]
            context_docs = batch["topk_documents"][i]
            context = " ".join(context_docs)
            input_text = f"question: {question}\ncontext: {context}" #provide a detailed explanation ...
            batch_inputs.append(input_text)
            references.append(batch["answer"][i])

        inputs = tokenizer(
            batch_inputs,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        labels = tokenizer(
            references,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_length,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3,
                length_penalty=0.7,
                #temperature=0.7
            )

        decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_predictions.extend(decoded_preds)
        all_references.extend(references)
        all_inputs.extend(batch_inputs)

    predictions = [pred.strip() for pred in all_predictions]
    references = [ref.strip() for ref in all_references]

    formatted = []
    print(f"FORMATTED ------------ \n\n\n")
    for i, (pred, ref) in enumerate(zip(predictions, references)):
        temp = {
            "id": str(i),
            "prediction_text": pred,
            "answers": {"text": [ref], "answer_start": [0]}
        }
        print(f"{temp}")
        formatted.append({
            "id": str(i),
            "prediction_text": pred,
            "answers": {"text": [ref], "answer_start": [0]}
    })

    results = {}

    em_score = exact_match_metric.compute(predictions=predictions, references=references)
    #squad = squad.compute(predictions=predictions, references=formatted)
    rouge_score = rouge_metric.compute(predictions=predictions, references=references, use_stemmer=True)
    bertscore = bertscore_metric.compute(predictions=predictions, references=references, lang="en")
    perplexity = compute_perplexity(model, tokenizer, inputs, labels, device)

    results["exact_match"] = em_score["exact_match"]
    #results["SQuAD"] = squad["exact"]
    results["rougeL"] = rouge_score["rougeL"]
    results["bertscore_f1"] = np.mean(bertscore["f1"])
    results["perplexity"] = perplexity

    idxs = rd.sample(range(len(predictions)), 10)

    for i in idxs:
        pred = predictions[i]
        ref = references[i]
        q = test_dataset["question"][i]
    
        print(f"Question: {q}\nReference: {ref}\nGenerated: {pred}\n\n")
    
    return results

In [ ]:
#t = pd_to_hf_ds(topk_df_test, tokenizer)

base_metrics = evaluate_model(model, tokenizer, topk_ds_test)

print("\nMetrics:")
for k, v in base_metrics.items():
    print(f"{k}: {v:.4f}")

Evaluating:   0%|          | 0/254 [00:00<?, ?it/s]

question: Is the following statement correct or not? Say true if it's correct; otherwise say false.
The city saw a decrease in the percentage of the population below the poverty line from 2000 to 2010.
context: males. The median income for a household in the city was $29,380, and the median income for a family was $32,392. Males had a median income of $30,015 versus $22,215 for females. The per capita income for the city was $14,149. About 14.9% of families and 16.2% of the population were below the poverty line, including 22.8% of those under age 18 and 11.1% of those age 65 or over. As of the census of 2010, there were 6,397 people, 2,307 households, and 1,520 families residing in the city. The population density was 999.5 people per square mile (385.4/km²). There food basket using prices for the year 2000 instead of from nearly a half century earlier, it was found that the poverty line should actually be 200% higher than the official level being used by the government in that year. 

Evaluating:   0%|          | 1/254 [00:42<2:58:26, 42.32s/it]

question: Choose the best option for the question below:

What is a significant impact of tourism on local economies such as that of Clatsop County?
A. Tourism leads to the decline of local culture and traditions.
B. Tourism significantly increases traffic congestion and pollution.
C. Tourism creates jobs and supports local businesses.
D. Tourism results in reduced availability of natural resources.
context: crime, pollution, prostitution, and a decline in social stability” as well as growth of capitalist values and a consumer culture. The demonstration effect was introduced to tourism when researchers were looking into the effects of social influences from tourism on local communities. The demonstration effect argues that local inhabitants copy the behavioral patterns of tourists. There are a number of social, economic and behavioral reasons as to why the demonstration effect comes into play. One economic and social reason is that locals copy the consumption patterns of those higher u

Evaluating:   1%|          | 2/254 [01:32<3:17:50, 47.11s/it]

...

question: Pick the right choice from the options provided below:

What historical figure is Dorset famously associated with?
A. Queen Elizabeth I
B. William Shakespeare
C. Thomas Hardy
D. Isaac Newton
context: Chibnall's decision to set the show on the Jurassic Coast also helped him generate more ideas for the show and tighten the writing. For example, Dorset's most famous native son, poet and author Thomas Hardy, lent his last name to one of the main characters (DI Alec Hardy). Hardy's use of the term "Wessex" was used to name the fictional Wessex Police, and character Jack Marshall reads the Hardy novel "Jude the Obscure". The series' name also came from the Dorset setting. Chibnall invented the name "Broadchurch" based on two settlements in Dorset: "I thought a lot about the literary National Trust. The visitor centre opened in September 2014. Thomas Hardy's Cottage Thomas Hardy's Cottage, in Higher Bockhampton, Dorset, is a small cob and thatch building that is the birthplace of th

Evaluating:  13%|█▎        | 34/254 [27:17<2:31:50, 41.41s/it]

question: Describe the process and conditions for forming the lactam 5-cyano-2-piperidone from 2-methyleneglutaronitrile.
context: Polytrimethylene terephthalate Polytrimethylene terephthalate (PTT), is a polyester synthesized and patented in 1941. It is produced by a method called condensation polymerization or transesterification. The two monomer units used in producing this polymer are: 1,3-propanediol and terephthalic acid or dimethyl terephthalate. Similar to polyethylene terephthalate, the PTT is used to make carpet fibers. PTT's value as a commercial polymer has improved due to more economical and efficient methods to produce 1,3-propanediol in the 1980s by Degussa, via acrolein, and Shell via the hydroformylation of ethylene oxide. DuPont has successfully commercialized the production of this polymer via 1,3-propanediol obtained by fermentation. reactions with alkoxides, aryloxides, amines or organometallic reagents. Because many different reagents can participate in this macro

Evaluating:  14%|█▍        | 35/254 [27:48<2:20:45, 38.57s/it]

question: Answer the following question by selecting one of the options:

Which process involves a chemical change?
A. Cutting fruit into slices.
B. Boiling water to make tea.
C. Burning wood in a fireplace.
D. Dissolving sugar in water.
context: destructive distillation is to coal. Historically the process of destructive distillation and other forms of pyrolysis led to the discovery of many chemical compounds or elucidation of their structures before contemporary organic chemists had developed the processes to synthesise or specifically investigate the parent molecules. It was especially in the early days that investigation of the products of destructive distillation, like those of other destructive processes, played parts in enabling chemists to deduce the chemical nature of many natural materials. Well known examples include the deduction of the structures of pyranoses and furanoses. The process of pyrolysis can be conducted Kraft process The kraft process (also known as kraft pulpi

### LoRA FT

In [ ]:
def preprocess_function(examples, tokenizer, max_source_length=512, max_target_length=512):
    inputs = []
    for question, documents in zip(examples["question"], examples["topk_documents"]):
        context = " ".join(documents)
        #inputs.append(f"question: {question} context: {context}")

        inputs.append(f"Question:\n{question}\n\nContext:\n{context}\n\nAnswer: ")
    
    model_inputs = tokenizer(
        inputs, 
        max_length=max_source_length, 
        truncation=True, 
        padding="max_length"
    )
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["answer"], 
            max_length=max_target_length, 
            truncation=True, 
            padding="max_length"
        )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
def create_lora_model(model):
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        inference_mode=False,
        r=8, 
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=["q", "k", "v"]#, "o", "Linear"]
    )
    
    return get_peft_model(model, lora_config)

In [ ]:
def pd_to_hf_ds(dataset, tokenizer):
    #hf_ds = Dataset.from_pandas(dataset)
    tokenized_dataset = dataset.map(
        lambda x: preprocess_function(x, tokenizer), batched=True
    )
    return tokenized_dataset

In [ ]:
model_name = "google/flan-t5-large" # the model name
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = create_lora_model(model)
model.print_trainable_parameters()

topk_hf_train = pd_to_hf_ds(topk_ds_train, tokenizer)
topk_hf_val = pd_to_hf_ds(topk_ds_val, tokenizer)

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/train",
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    learning_rate=5e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=2,
    predict_with_generate=True,
    bf16=True, #fp16
    gradient_accumulation_steps=4, #4
    report_to="tensorboard",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=topk_hf_train,
    eval_dataset=topk_hf_val,
    tokenizer=tokenizer,
)

In [ ]:
trainer.train()

model.save_pretrained("t5-lora-qa")
tokenizer.save_pretrained("t5-lora-qa")

In [ ]:
ft_metrics = evaluate_model(model, tokenizer, topk_ds_test)

print("\nMetrics:")
for k, v in ft_metrics.items():
    print(f"{k}: {v:.4f}")

In [ ]:
from peft import PeftModel

In [ ]:
base_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

model = PeftModel.from_pretrained(base_model, "/kaggle/input/checkpoint")

In [ ]:
model.train()  

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/train",
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    learning_rate=5e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=2,
    predict_with_generate=True,
    bf16=True, #fp16
    gradient_accumulation_steps=4, #4
    report_to="tensorboard",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=topk_hf_train,
    eval_dataset=topk_hf_val,
    tokenizer=tokenizer,
)

trainer.train(resume_from_checkpoint="/kaggle/input/checkpoint") 

In [ ]:
trainer.train(resume_from_checkpoint=True) 